# Rollout Collection: Workstream 1 scaffold vs. the real gemma-4-31b-it-qat-w4a16-ct

Run our actual `submission/` scaffold (not the sample submission) against a locally-pre-verified-resolvable subset of official tasks, through the real 31B served on this notebook's GPU(s).

**Runs 1-4** iterated on `eval_config.yaml`'s per-task ceiling numbers and found each tighter config mostly traded "ran out of budget" for "ran out of budget less often" without addressing why tasks needed so much room.

**Research pivot**: two public Kaggle notebooks for this competition (pulled via the API) revealed the only two public submissions that actually score ship no `eval_config.yaml` at all. Four root causes found and fixed in `submission/`: `search_similar_code`'s context-overflow risk (removed the tool), pytest output-truncation blindness (fixed), the graph's async blind spot (warned), and `skip_summarization: true`'s wasted-turn tax (fixed).

**Run 6** (eval_config.yaml removed, fixes applied): on the exact same 10 task IDs as runs 3-4, **zero attempts hit any ceiling** -- every task previously truncated by turns/time now ran to genuine completion. One attempt (`rich_3944`, different task set, run 4) crashed with `litellm.ContextWindowExceededError` from `max_output_tokens: 16384` reserving half the 32,768-token context -- fixed by lowering to 8192. But natural per-task time ranged 184s-1483s (avg ~637s/10.6min); extrapolated across ~120 hidden tasks that's ~21h, over the 12h shared budget.

**Run 7** (this run): same 10 tasks as run 6, `eval_config.yaml` reintroduced with ONLY `max_time_minutes: 6` set (no turns/tool-calls cap, since those weren't the real problem once the root causes were fixed) -- chosen from run 6's actual timing distribution (7/10 tasks finished under 10 minutes naturally). Tests whether a data-informed time-only cap keeps completions genuine while bounding the few real outliers.

## 1. Environment Configuration and Package Installation

In [ ]:
import glob
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        os.symlink(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
print('Wheelhouse installation complete.')

## 2. Our Submission Scaffold and Task Selection

The submission dataset (`noobhunterjay/gemma4-agent-submission`) mirrors the git-tracked `submission/` directory in the project repo exactly, including the graph-tool-usage and eval_config.yaml fixes made after submission 56682704 timed out -- NOT the competition's `sample_submission/`. Rather than guessing the exact Kaggle mount-path convention (it has changed between competitions before), we search for it by content.

In [ ]:
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Find our submission dataset under /kaggle/input by content (agent.yaml naming ours
# 'swe_scaffold_v1'), rather than assuming a specific mount-path convention.
candidates = [
    p.parent for p in Path('/kaggle/input').rglob('agent.yaml')
    if 'swe_scaffold_v1' in p.read_text(encoding='utf-8')
]
assert candidates, 'Could not find our submission dataset under /kaggle/input -- check dataset_sources in kernel-metadata.json'
OUR_SUBMISSION_SRC = candidates[0]
print(f'Found our submission scaffold at: {OUR_SUBMISSION_SRC}')

AGENT_DIR = WORKING_DIR / 'submission'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
shutil.copytree(OUR_SUBMISSION_SRC, AGENT_DIR)

TASKS_PATH = DATA_DIR / 'tasks.jsonl'
all_tasks = load_tasks(TASKS_PATH)
by_id = {t.instance_id: t for t in all_tasks}

# Run 5: a 10-task subset of runs 3-4's 20-task set (same instance_ids, for a clean
# 3-way comparison), trimmed from 20 because harness defaults allow up to 60 minutes
# per task -- 20 tasks at a 60-minute worst case risks a very long/expensive run,
# and we only need enough signal to tell whether the root-cause fixes (not the
# ceiling number) actually reduce how often tasks get stuck.
SMOKE_TEST_IDS = [
    'rich_4075', 'rich_4006', 'rich_3718', 'rich_3934', 'rich_3953',
    'fastapi_15589', 'fastapi_15280', 'fastapi_14986',
    'requests_7427', 'requests_7315',
]
SAMPLE_TASKS = [by_id[i] for i in SMOKE_TEST_IDS if i in by_id]

print(f'Agent directory: {AGENT_DIR}')
print(f'Loaded {len(all_tasks)} total tasks; running validation batch on {len(SAMPLE_TASKS)}:')
for t in SAMPLE_TASKS:
    print(f'  - {t.instance_id} ({t.repo} @ {t.base_commit[:8]})')

## 3. Start vLLM Server (real `gemma-4-31b-it-qat-w4a16-ct`)

In [ ]:
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
from swegemma.models.discovery import validate_single_declared_model

litellm.drop_params = True

TARGET_MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')
INFERENCE_API_KEY = 'EMPTY'

# Validate single base model and discover any PEFT LoRA adapters (none yet -- Workstream 3)
declared_model = validate_single_declared_model(AGENT_DIR)
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=True,
    max_loras=8,
    max_lora_rank=128,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

models = server_instance.create_model_registry(
    aliases=[declared_model, TARGET_MODEL_NAME],
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)

## 4. Rollout Collection: run the agent, verify, keep only resolved trajectories

Mirrors `scripts/collect_rollouts.py`'s logic (same two-phase `Evaluator.evaluate_task` the real scorer uses), adapted for the notebook's `subprocess` sandbox instead of local Docker. Every attempt (resolved or not) is recorded; the ATIF trace for each is saved so resolved ones can be turned into SFT training data afterward.

In [ ]:
import asyncio
import concurrent.futures
import json
import time
import yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator
from swegemma.harness.verification import save_trace_artifact


def run_sync(coro_fn, **kwargs):
    """Run an async coroutine synchronously inside a notebook event loop."""
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(coro_fn(**kwargs))).result()
    return asyncio.run(coro_fn(**kwargs))


# Run 5: no eval_config.yaml shipped (removed from the submission scaffold) -- fall
# back entirely to harness defaults (60 min / 100 tool calls / 500 turns per task,
# per HARNESS_README.md), matching what the two actually-scoring public community
# notebooks do, rather than our own guessed per-task ceilings.
eval_config_file = AGENT_DIR / 'eval_config.yaml'
budget_overrides = {}
if eval_config_file.exists():
    raw_eval_cfg = yaml.safe_load(eval_config_file.read_text(encoding='utf-8'))
    eval_section = raw_eval_cfg.get('evaluation', raw_eval_cfg)
    if 'timeout_seconds' in eval_section:
        budget_overrides['timeout_seconds'] = int(eval_section['timeout_seconds'])
    if 'max_tool_calls' in eval_section:
        budget_overrides['max_tool_calls'] = int(eval_section['max_tool_calls'])
    if 'max_time_minutes' in eval_section:
        budget_overrides['max_time_minutes'] = float(eval_section['max_time_minutes'])
    turns_raw = eval_section.get('max_turns', eval_section.get('max_llm_calls'))
    if turns_raw is not None:
        budget_overrides['max_turns'] = int(turns_raw)
    print(f'Found eval_config.yaml -- overrides: {budget_overrides}')
else:
    print('No eval_config.yaml found -- using pure harness defaults (60 min / 100 tool calls / 500 turns per task).')

RESULTS_DIR = WORKING_DIR / 'rollouts'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SAMPLES_PER_TASK = 1

limits, gen_constraints = build_submission_limits()
eval_config = EvalConfig(
    tasks_path=TASKS_PATH,
    snapshots_dir=DATA_DIR / 'snapshots',
    results_dir=RESULTS_DIR,
    submission_dir=AGENT_DIR,
    models=models,
    sandbox='subprocess',
    limits=limits,
    generation_constraints=gen_constraints,
    adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=15,
        overlap_size=2,
        token_threshold=14336,
        event_retention_size=5,
    ),
    graph_dir=str(DATA_DIR / 'graphs'),
    embeddings_dir=str(DATA_DIR / 'embeddings'),
    wheels_dir=DATA_DIR / 'wheels',
    verbose=False,
    **budget_overrides,
)
print(f'Effective per-task budget: time_minutes={eval_config.budget.time_minutes}, tool_calls={eval_config.budget.tool_calls}, turns={eval_config.budget.turns}, command_timeout_seconds={eval_config.harness.command_timeout_seconds}')

evaluator = Evaluator(eval_config)
records = []
attempt = 0
total_attempts = len(SAMPLE_TASKS) * SAMPLES_PER_TASK

for task in SAMPLE_TASKS:
    for sample_idx in range(SAMPLES_PER_TASK):
        attempt += 1
        print(f'[{attempt}/{total_attempts}] {task.instance_id} sample {sample_idx}...')
        start = time.perf_counter()
        result = run_sync(
            evaluator.evaluate_task,
            task=task,
            task_index=attempt,
            total_tasks=total_attempts,
        )
        trace_path = save_trace_artifact(
            getattr(result, 'trace', None), RESULTS_DIR, f'{task.instance_id}__s{sample_idx}'
        )
        record = {
            'instance_id': task.instance_id,
            'repo': task.repo,
            'sample_idx': sample_idx,
            'resolved': bool(result.resolved),
            'agent_patch': result.agent_patch or '',
            'error': result.error,
            'test_exit_code': result.test_exit_code,
            'duration_seconds': result.duration_seconds,
            'tool_calls': result.tool_calls,
            'total_llm_calls': getattr(result, 'total_llm_calls', None),
            'trace_path': trace_path,
        }
        records.append(record)
        status = 'KEPT (resolved)' if record['resolved'] else 'discarded (failed)'
        print(f'  -> {status} in {time.perf_counter() - start:.1f}s' + (f' -- {record["error"]}' if record['error'] else ''))

        with open(RESULTS_DIR / 'rollouts.jsonl', 'a', encoding='utf-8') as f:
            f.write(json.dumps(record, default=str) + '\n')

n_kept = sum(r['resolved'] for r in records)
print(f'\n{n_kept}/{len(records)} attempts resolved on the REAL 31B (kept as SFT-candidate trajectories)')

by_repo = {}
for r in records:
    by_repo.setdefault(r['repo'], []).append(r['resolved'])
for repo, flags in sorted(by_repo.items()):
    print(f'  {repo}: {sum(flags)}/{len(flags)}')

print(f'Results written to {RESULTS_DIR / "rollouts.jsonl"} (download via notebook output)')